# Aim 1 replication, datasets 2 and 3 — Kang 2018 and Randolph 2021

**Finding to test (Parse, corrected pipeline):** in lymphocytes, pairs of typical donors differ little in response *shape*, while pairs involving a high-baseline-interferon donor differ much more in shape.

| Dataset | Donors | Stimulus | Why it helps |
|---|---|---|---|
| **Kang et al. 2018** (GSE96583, batch 2) | 8 lupus patients | IFN-beta vs control, 6 h | Cells of all donors were **pooled and stimulated together**, so well noise and processing date are shared by all donors and cannot create donor differences. Baseline interferon varies naturally in lupus. |
| **Randolph et al. 2021** (GSE162632) | ~90 | Influenza A vs mock, 6 h | Many donors. Design (pooling, batches) must be checked first. |

This notebook only downloads and inspects the data. The analysis is in `aim1_replication_kang_analysis.ipynb`.

In [ ]:
import os, subprocess, re
import numpy as np, pandas as pd

KANG_DIR = "data/kang2018"
RAND_DIR = "data/randolph2021"
for d in [KANG_DIR, RAND_DIR]: os.makedirs(d, exist_ok=True)
RESULTS_DIR = "results/aim1_replication2"
os.makedirs(RESULTS_DIR, exist_ok=True)

def list_ftp(url):
    # file names in a GEO FTP-over-HTTPS directory (relative links only; page-footer links are skipped)
    html = subprocess.run(["curl", "-sL", url], capture_output=True, text=True).stdout
    names = re.findall(r'href="([^"?]+)"', html)
    return sorted({n for n in names if "://" not in n and not n.startswith("/") and not n.endswith("/")})

def remote_size(url):
    head = subprocess.run(["curl", "-sIL", url], capture_output=True, text=True).stdout
    sizes = [int(l.split(":")[1]) for l in head.splitlines() if l.lower().startswith("content-length")]
    return sizes[-1] if sizes else None

def download(url, out):
    remote = remote_size(url)
    if os.path.exists(out) and remote is not None and os.path.getsize(out) == remote:
        print(f"{os.path.basename(out)}: already complete"); return
    subprocess.run(["curl", "-L", "-C", "-", "--retry", "5", "--progress-bar", "-o", out, url])
    size = os.path.getsize(out) if os.path.exists(out) else 0
    status = "OK" if remote is None or size == remote else f"INCOMPLETE ({size:,} of {remote:,} bytes)"
    print(f"{os.path.basename(out)}: {size / 1e6:.1f} MB {status}")

## 1. Kang 2018: list and download

In [ ]:
KANG_SERIES = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE96nnn/GSE96583/suppl/"
KANG_GSM = {"ctrl": "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2560nnn/GSM2560248/suppl/",
            "stim": "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2560nnn/GSM2560249/suppl/"}

series_files = list_ftp(KANG_SERIES)
print("Series files:", series_files)
gsm_files = {k: list_ftp(u) for k, u in KANG_GSM.items()}
print("Sample files:", gsm_files)

# batch-2 genes + metadata from the series, matrix + barcodes from the two samples
for f in series_files:
    if "batch2" in f:
        download(KANG_SERIES + f, os.path.join(KANG_DIR, f))
for k, files in gsm_files.items():
    for f in files:
        download(KANG_GSM[k] + f, os.path.join(KANG_DIR, f))
print(sorted(os.listdir(KANG_DIR)))

## 2. Kang 2018: inspect metadata

In [ ]:
meta_files = [f for f in os.listdir(KANG_DIR) if "batch2" in f and ("tsne" in f or "df" in f)]
print("metadata candidates:", meta_files)
km = pd.read_csv(os.path.join(KANG_DIR, meta_files[0]), sep="\t")
print(km.head(), "\n\ncolumns:", km.columns.tolist(), "\nindex example:", km.index[:3].tolist())
for col in km.columns:
    if km[col].nunique() < 30:
        print(f"\n{col}:", km[col].value_counts().to_dict())
bc = {k: pd.read_csv(os.path.join(KANG_DIR, [f for f in os.listdir(KANG_DIR) if f.startswith(g) and "barcodes" in f][0]),
                     header=None)[0].head(3).tolist()
      for k, g in [("ctrl", "GSM2560248"), ("stim", "GSM2560249")]}
print("\nbarcode examples:", bc)

## 3. Randolph 2021: list files (download after checking sizes and format)

In [ ]:
RAND_SERIES = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE162nnn/GSE162632/suppl/"
rand_files = list_ftp(RAND_SERIES)
print("Series files:", rand_files)
for f in rand_files:
    head = subprocess.run(["curl", "-sIL", RAND_SERIES + f], capture_output=True, text=True).stdout
    size = [l for l in head.splitlines() if l.lower().startswith("content-length")]
    print(f, size[-1] if size else "")

## 4. Randolph 2021: archive contents
The series has only a 0.8 GB RAW archive (plus bulk tables). Its file list shows the format before downloading.

In [ ]:
# Randolph 2021: what is inside the RAW archive?
download(RAND_SERIES + "filelist.txt", os.path.join(RAND_DIR, "filelist.txt"))
fl = pd.read_csv(os.path.join(RAND_DIR, "filelist.txt"), sep="\t")
pd.set_option("display.max_rows", 200, "display.width", 200)
fl